# Notebook 5 - Deep-Dive Explanation Agent

The **second reasoning agent** in the system. The operational chain (notebook 3)
*classifies* flights; this agent *explains* the HIGH-flagged ones. Distinct
objective, own context (one flight's raw signals + the model's attribution), and it
produces the engineer-readable hypothesis that goes into the maintenance queue in
place of the static evidence string.

**Honest split:** three tools *measure* deterministically, the LLM does the *judgment*.

| tool | question | kind |
|---|---|---|
| `segment_flight` | which phase / when? | deterministic |
| `attribute_prediction` | which channels did the *model* rely on? (occlusion) | deterministic |
| `localize_anomaly` | which cylinder deviates, in which phase, how far past normal? (raw signal) | deterministic |
| **LLM synthesis** | reconcile attribution vs raw signal into a hedged hypothesis | **reasoning** |

## 1 - Config

All settings here, same pattern as notebook 4's top cell.

In [18]:
from dotenv import load_dotenv
import os, sys, json
from pathlib import Path
import numpy as np, pandas as pd
load_dotenv()

sys.path.insert(0, "../scripts")          # repo scripts/ (realdata, featurize_spec, deep_dive_agent, ...)

# --- LLM endpoint (same pattern as notebook 4) ---
LOCAL      = False
BASE_URL   = "http://localhost:8000/v1" if LOCAL else os.getenv("API_URL")
MODEL_NAME = "Qwen/Qwen3-Coder-30B-A3B-Instruct" if LOCAL else os.getenv("API_MODEL_NAME")
API_KEY    = "not-needed" if LOCAL else os.getenv("RIT_API_KEY")

# --- paths (match notebook 4 artifacts) ---
DATA_DIR = "../data"
SPEC     = f"{DATA_DIR}/best_spec.json"          # discovered feature spec (nb4)
MODEL    = f"{DATA_DIR}/c28_model.joblib"        # {'model','feature_columns'} bundle (nb4)
DEMO_DIR = f"{DATA_DIR}/c28_demo/flights"                # directory of per-flight CSVs (export_flights_to_dir)
METADATA = f"{DEMO_DIR}/../metadata.csv"            # metadata CSV export wrote alongside; adjust if named differently
FILE_COL, LABEL_COL, GROUP_COL = "filename", "label", "plane_id"   # columns in METADATA

# --- featurizer: MUST match the trained model ---
USE_SPEC_FEATURIZER = True    # nb4 trains the model on the discovered spec; False -> baseline summary_stats spec

# --- deep-dive selection + tools ---
HIGH_THRESHOLD = 0.50         # p_maintenance at/above which a flight gets a deep dive
MAX_DEEP_DIVES = 25           # cap on LLM synthesis calls per run
SAMPLE_HZ      = 1.0
EGT_THRESHOLD  = 30.0         # degF phase-contrast to flag a hot/cold cylinder (EGT, normal spread ~40)
CHT_THRESHOLD  = 20.0         # degF (CHT corroboration, normal spread ~19)
LOC_THRESHOLDS = {"EGT": EGT_THRESHOLD, "CHT": CHT_THRESHOLD}

print("config loaded | LOCAL =", LOCAL, "| endpoint:", "local vLLM" if LOCAL else "RIT API",
      "| featurizer:", "spec" if USE_SPEC_FEATURIZER else "baseline")

config loaded | LOCAL = False | endpoint: RIT API | featurizer: spec


## 2 - Load the model bundle

In [19]:
import joblib
bundle = joblib.load(MODEL)
assert "model" in bundle and "feature_columns" in bundle, "bundle needs model + feature_columns"
clf, FEATURE_COLS = bundle["model"], bundle["feature_columns"]
print(f"model: {type(clf).__name__} | {len(FEATURE_COLS)} features")

model: HistGradientBoostingClassifier | 412 features


## 3 - The feature spec

Featurization is done by the pipeline's own runtime featurizer,
`featurize_spec.build_feature_table_from_dir`, so the columns match
`bundle['feature_columns']` exactly (its docstring guarantees parity with
training). All we choose here is *which spec* to run:

- `USE_SPEC_FEATURIZER=True`  -> the discovered spec from notebook 4 (`best_spec.json`)
- `USE_SPEC_FEATURIZER=False` -> a plain `summary_stats` spec over all channels, which
  reproduces the baseline 12-stat features (same `{channel}__{stat}` names as `train_real_baseline`)

Either way it's one code path; the flag only picks the spec.

In [20]:
SENSORS = ["volt1","volt2","amp1","amp2","FQtyL","FQtyR","E1 FFlow","E1 OilT","E1 OilP","E1 RPM",
           "E1 CHT1","E1 CHT2","E1 CHT3","E1 CHT4","E1 EGT1","E1 EGT2","E1 EGT3","E1 EGT4",
           "OAT","IAS","VSpd","NormAc","AltMSL"]

if USE_SPEC_FEATURIZER:
    spec = json.load(open(SPEC))
    print("discovered spec:", spec.get("spec_id", "(unnamed)"), "|",
          len(spec.get("features", [])), "feature entries")
else:
    spec = {"features": [{"transform": "summary_stats", "channels": SENSORS}]}
    print("baseline summary_stats spec over", len(SENSORS), "channels")

discovered spec: spec_474e3e9f3b | 5 feature entries


## 4 - Featurize the demo batch + classify

`build_feature_table_from_dir` reads the metadata + per-flight CSVs and applies the spec. The table is indexed by `filename`, which we reuse to load raw frames for the tools.

In [21]:
from featurize_spec import build_feature_table_from_dir

table = build_feature_table_from_dir(DEMO_DIR, METADATA, spec,
                                     file_col=FILE_COL, label_col=LABEL_COL, group_col=GROUP_COL)
table = table.set_index(FILE_COL)

X = table.reindex(columns=FEATURE_COLS, fill_value=0.0).fillna(0.0)
table["p_maintenance"] = clf.predict_proba(X.values)[:, 1]

high = table["p_maintenance"][table["p_maintenance"] >= HIGH_THRESHOLD] \
          .sort_values(ascending=False).head(MAX_DEEP_DIVES)
print(f"{len(table)} flights scored | {len(high)} flagged HIGH (p >= {HIGH_THRESHOLD})")
high

200 flights scored | 25 flagged HIGH (p >= 0.5)


filename
flight_1240.csv    0.983345
flight_1686.csv    0.971418
flight_947.csv     0.967698
flight_78.csv      0.960536
flight_595.csv     0.957496
flight_690.csv     0.953768
flight_2113.csv    0.948834
flight_1379.csv    0.938496
flight_781.csv     0.932923
flight_2182.csv    0.931192
flight_1724.csv    0.930661
flight_147.csv     0.925767
flight_1114.csv    0.913122
flight_1412.csv    0.913052
flight_979.csv     0.911154
flight_1696.csv    0.909667
flight_556.csv     0.908351
flight_614.csv     0.902913
flight_134.csv     0.889450
flight_1245.csv    0.881674
flight_1362.csv    0.881590
flight_1051.csv    0.876009
flight_1094.csv    0.874304
flight_181.csv     0.864682
flight_1295.csv    0.855843
Name: p_maintenance, dtype: float64

## 5 - Background reference for occlusion

Median feature vector to occlude toward in attribution ("what if this channel looked typical?").

In [22]:
from attribute import compute_background
background = compute_background(table, FEATURE_COLS)   # optional: normal_mask=(table['label']==0)
print("background ready")

background ready


## 6 - LLM synthesis endpoint

Uses the top-cell `BASE_URL` / `MODEL_NAME` / `API_KEY`. If `BASE_URL` is unset, `chat_fn=None` and the agent falls back to its deterministic template so the notebook still produces a queue.

In [23]:
import urllib.request

def make_chat_fn(base_url, model_name, api_key):
    if not (base_url and model_name):
        return None
    def chat_fn(system, user):
        body = json.dumps({"model": model_name, "temperature": 0.2,
                           "messages": [{"role":"system","content":system},
                                        {"role":"user","content":user}]}).encode()
        req = urllib.request.Request(base_url.rstrip("/") + "/chat/completions", data=body,
                                     headers={"Content-Type":"application/json",
                                              **({"Authorization": f"Bearer {api_key}"} if api_key else {})})
        with urllib.request.urlopen(req, timeout=120) as r:
            return json.loads(r.read())["choices"][0]["message"]["content"]
    return chat_fn

chat_fn = make_chat_fn(BASE_URL, MODEL_NAME, API_KEY)
print("LLM synthesis:", "live endpoint" if chat_fn else "template fallback (set BASE_URL/MODEL_NAME to enable)")

LLM synthesis: live endpoint


## 7 - Deep-dive each HIGH flight -> maintenance queue

For each flagged flight: read its raw frame from the dir, take its feature row from the table, run the three tools + synthesis. The explanation replaces the static `evidence` string.

In [24]:
from deep_dive_agent import deep_dive

queue = []
for fn in high.index:
    df = pd.read_csv(Path(DEMO_DIR) / str(fn)).reset_index(drop=True)
    frow = table.loc[fn]                       # attribute_prediction reindexes to FEATURE_COLS
    dd = deep_dive(df, frow, bundle, background, chat_fn=chat_fn, flight_id=str(fn),
                   sample_hz=SAMPLE_HZ, loc_thresholds=LOC_THRESHOLDS)
    queue.append(dd)

print(f"deep-dived {len(queue)} flights")

deep-dived 25 flights


## 8 - The maintenance queue

In [25]:
for dd in sorted(queue, key=lambda d: -d["p_maintenance"]):
    print(f"\n=== {dd['flight']} | p_maintenance={dd['p_maintenance']:.2f} ===")
    print(dd["explanation"])


=== flight_1240.csv | p_maintenance=0.98 ===
The model flags this flight for inspection with high confidence (p_maintenance = 0.9833), indicating a potential maintenance issue. The occlusion attribution points to the E1 OilP channel as the most significant factor, though this channel was not localized by the raw signal analysis. The raw signal analysis did identify a cross-cylinder imbalance localized to E1 EGT2 during the descent phase, with an excess deviation of -45.4°C, but this does not align with the model's primary attribution to E1 OilP. Since the model's key attribution and the raw signal localization disagree on both channel and cylinder, the cause remains uncertain and requires further investigation.

=== flight_1686.csv | p_maintenance=0.97 ===
The model flags this flight for inspection with high confidence (p_maintenance = 0.9714), primarily attributing the risk to the E1 OilP channel, which showed the largest probability drop (0.1909). However, the raw signal localizatio

## 9 - Inspect one flight's full evidence

The deterministic measurements the LLM reasoned over - every explanation is auditable back to numbers.

In [26]:
if queue:
    ev = queue[0]["evidence"]
    print(f"flight {queue[0]['flight']}\n")
    print("segmentation:", json.dumps(ev["segmentation"], indent=2))
    print("\ntop attribution (occlusion):")
    for a in ev["attribution_top"][:5]:
        print(f"  {a['channel']:12s} prob_drop={a['prob_drop']:+.3f}")
    print("\nlocalization (raw-signal cross-cylinder):")
    for f in ev["localization"]:
        print(f"  {f['channel']} cyl{f['cylinder']} {f['direction']} "
              f"excess {f['excess']:+.0f}degF in {f['worst_phase']} "
              f"(corroborated={f.get('corroborated_by_other_group')})")

flight flight_1240.csv

segmentation: {
  "n_takeoffs": 5,
  "n_landings": 5,
  "airborne_s": 4805.0,
  "phase_seconds": {
    "ground": 1494.0,
    "climb": 1030.0,
    "cruise": 2987.0,
    "descent": 930.0
  }
}

top attribution (occlusion):
  E1 OilP      prob_drop=+0.329
  E1 CHT3      prob_drop=+0.006
  E1 CHT4      prob_drop=+0.003
  E1 CHT1      prob_drop=+0.001
  E1 EGT4      prob_drop=+0.001

localization (raw-signal cross-cylinder):
  E1 EGT2 cyl2 cold excess -45degF in descent (corroborated=False)
